# Pruebas A/B y Experimentación en Entornos Profesionales

## Objetivo de la sesión

Comprender y aplicar el proceso profesional de experimentación A/B, incluyendo sus etapas, fundamentos estadísticos y la interpretación de resultados para la toma de decisiones empresariales, especialmente en fijación de precios y optimización de KPIs.

---

## 1. ¿Qué es una Prueba A/B?

Una **prueba A/B** es un experimento controlado en el que dos variantes (A y B) de una variable (por ejemplo, un precio, diseño, o algoritmo) se comparan para determinar cuál tiene mejor desempeño respecto a uno o varios KPIs. Es fundamental para validar hipótesis antes de realizar cambios significativos en el negocio.

* **Ejemplos de uso:**

  * Marketing digital (copy, banners, landing pages).
  * Cambios de precio.
  * Nuevas funcionalidades en productos digitales.
  * Experimentos con modelos de machine learning.

---

## 2. Etapas Profesionales de una Prueba A/B

> Cada etapa incluye preguntas y cálculos clave para asegurar el rigor y éxito experimental.

### **2.1. Planteamiento del problema y objetivos**

* ¿Cuál es la pregunta de negocio?
* ¿Qué cambio se quiere evaluar? ¿Por qué?
* ¿Qué KPI(s) medirán el éxito?

#### Ejemplo:

> “¿Incrementar el precio de $10 a $12 reduce conversiones, pero mejora ingresos totales y/o LTV?”

### **2.2. Definición de hipótesis**

* **Hipótesis nula ($H_0$):** El cambio no tiene efecto sobre el KPI.
* **Hipótesis alterna ($H_1$):** El cambio tiene efecto sobre el KPI.

#### Ejemplo:

* $H_0$: La tasa de conversión es igual en ambos precios.
* $H_1$: La tasa de conversión es distinta (menor o mayor) en el nuevo precio.

### **2.3. Diseño experimental y selección de métricas**

* ¿Qué variable se manipula (precio, diseño, etc)?
* ¿Qué métricas medirás? (conversión, ingresos, LTV, etc.)
* ¿Periodo del experimento?
* ¿Cómo se aleatorizarán los grupos?

#### Recomendaciones:

* Usa randomización para asignar usuarios/grupos.
* Evita contaminación cruzada entre grupos.



---

## 3. Conceptos Clave de la Estadística Experimental

Para tener un lenguaje común durante toda la sesión, aquí va una guía visual y breve de los conceptos clave. Puedes apoyarte en la siguiente infografía:

![Conceptos clave en pruebas estadísticas](https://www.researchgate.net/profile/Abdulkerim-Gok/publication/316927316/figure/fig3/AS:667699772391428@1536203439714/Left-Definitions-of-terminologies-in-a-statistical-test-Right-An-illustration-of-power.ppm)

* **Significancia ($alpha$):** Probabilidad de rechazar $H_0$ siendo verdadera (error tipo I).
* **Poder estadístico ($1-beta$):** Probabilidad de detectar un efecto real (evitar error tipo II).
* **P-value:** Probabilidad de observar un resultado igual o más extremo si $H_0$ es cierta.
* **Intervalo de confianza:** Rango donde se espera que caiga el verdadero valor poblacional.
* **Tamaño del efecto:** Magnitud práctica de la diferencia observada.
* **Mínima Diferencia Detectable:** El cambio más pequeño que te interesa detectar con tu experimento.


---

## 4. Cálculo del tamaño de la muestra y poder estadístico

El tamaño de la muestra garantiza que los resultados sean concluyentes (no “flukes”). El **poder estadístico** (usualmente ≥ 80%) es la probabilidad de detectar un efecto real.

**Factores a considerar:**

* Tasa base del KPI (p.ej. conversión).
* Diferencia mínima detectable (efecto mínimo relevante).
* Nivel de significancia (\$\alpha\$), comúnmente 0.05 (5%).

### **Código Python: Cálculo tamaño de muestra para tasas de conversión**

In [1]:
import numpy as np
from statsmodels.stats.power import zt_ind_solve_power

p1 = 0.05 # tasa de conversión base
diff = 0.01 # diferencia mínima detectable
alpha = 0.05 # nivel de significancia
power = 0.8 # poder estadístico

# Calcula el tamaño de muestra necesario por grupo
effect_size = (diff / np.sqrt(p1 * (1 - p1)))
n = zt_ind_solve_power(effect_size=effect_size, alpha=alpha, power=power, alternative='two-sided')
print(f'Tamaño de muestra necesario por grupo: {int(n)+1}')

Tamaño de muestra necesario por grupo: 7457


**Ref:** [statsmodels.stats.power](https://www.statsmodels.org/stable/generated/statsmodels.stats.power.tt_ind_solve_power.html)

### **Ejemplo con datos reales (conversiones):**

Usa los datos del ejemplo anterior:

In [2]:
# Simulación de resultados para dos precios
conversions_a = 48
trials_a = 1000
conversions_b = 30
trials_b = 900

from statsmodels.stats.proportion import proportions_ztest
count = np.array([conversions_a, conversions_b])
nobs = np.array([trials_a, trials_b])
stat, pval = proportions_ztest(count, nobs)
print(f"Z-stat: {stat:.3f}, P-value: {pval:.4f}")

Z-stat: 1.609, P-value: 0.1077


---

## 5. Ejecución y monitoreo del experimento

* Asegúrate de cumplir el tamaño de muestra calculado.
* No “detener temprano” (peeking): evita analizar resultados antes de tiempo.
* Monitorea métricas de salud del experimento (por ejemplo, tráfico, distribución, calidad de datos).


---

## 6. Análisis de resultados: ¿Qué pruebas estadísticas usar?

### 6.1 Para tasas de conversión (proporciones)

* **Z-test de proporciones** (si \$n\$ grande):

  * Ver ejemplo anterior.
* **Test exacto de Fisher** (si muestras pequeñas):

In [3]:
import scipy.stats as stats
# Ejemplo para grupos pequeños:
table = [[5, 20], [10, 20]]  # conversiones y no conversiones
d, p = stats.fisher_exact(table)
print(f"Odds ratio: {d:.3f}, P-value: {p:.4f}")

Odds ratio: 0.500, P-value: 0.3656


### 6.2 Para KPIs numéricos (ingresos, LTV, ticket)

* **Prueba t de dos muestras (t-test):** Si los datos son normales.
* **Welch's t-test:** Si varianzas o tamaños difieren.
* **Pruebas no paramétricas (Mann-Whitney):** Si los datos no son normales o hay outliers.

In [4]:
import numpy as np
import scipy.stats as stats
# Simula valores de LTV para dos grupos
ltv_a = [100, 110, 105, 98, 102, 95, 107]
ltv_b = [120, 125, 118, 130, 115, 122, 119]
t_stat, p_value = stats.ttest_ind(ltv_a, ltv_b, equal_var=False)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")

T-statistic: -6.9060, P-value: 0.0000


### 6.3. Interpretación de resultados

* Si $p$ < $alpha$ (típicamente 0.05), rechazas $H_0$ y el resultado es estadísticamente significativo.
* **Siempre reporta:**

  * El tamaño del efecto (diferencia entre medias/proporciones).
  * El intervalo de confianza (IC95%) usando `statsmodels.stats.weightstats.DescrStatsW` o similar.

In [5]:
import statsmodels.stats.api as sms
import numpy as np
# Para LTV (diferencia de medias y CI):
mean_diff = np.mean(ltv_b) - np.mean(ltv_a)
cm = sms.CompareMeans(sms.DescrStatsW(ltv_b), sms.DescrStatsW(ltv_a))
ci_low, ci_upp = cm.tconfint_diff(usevar='unequal')
print(f'Diferencia de medias: {mean_diff:.2f}, IC95%: ({ci_low:.2f}, {ci_upp:.2f})')

Diferencia de medias: 18.86, IC95%: (12.91, 24.81)


---

## 7. Decisión: ¿El resultado es significativo y relevante?

* Si el resultado es significativo (\$p\$ < 0.05), pero el tamaño del efecto es bajo, podría no ser relevante para negocio.
* Considera siempre la relevancia práctica y el contexto de negocio.
* Ajusta por pruebas múltiples si analizas varios KPIs (corrección de Bonferroni, FDR, etc).

---

## 8. Resumen gráfico del flujo de experimentación

```mermaid
graph TD;
  A[Planteamiento de pregunta y KPI] --> B[Definir hipótesis];
  B --> C[Diseño experimental y cálculo de muestra];
  C --> D[Ejecución y recolección de datos];
  D --> E[Análisis estadístico];
  E --> F[Decisión y comunicación de resultados];
```

---

## 9. Actividad práctica (en Jupyter)

1. Simula o usa un dataset real de conversiones o ventas para dos precios.
2. Calcula el tamaño de la muestra para tu caso.
3. Realiza la prueba estadística adecuada (usa los ejemplos de código de cada sección).
4. Interpreta los resultados: ¿debería la empresa implementar el nuevo precio?
5. Presenta tus resultados usando gráficos y argumentos claros.

---

## 10. Recursos y mejores prácticas

* [AB Testing Stats in Python – Towards Data Science](https://towardsdatascience.com/a-b-testing-with-python-e5964dd66143)
* [Google Optimize A/B Testing Guide](https://support.google.com/optimize/answer/6211930?hl=es)
* [Optimizely AB Testing Knowledge Base](https://www.optimizely.com/optimization-glossary/ab-testing/)
* [Documentación oficial de statsmodels](https://www.statsmodels.org/stable/index.html)
